# MAP5935 - Statistical Learning (Chapter 8 - Tree-Based Models)

**Prof. Christian Jäkel**

https://www.statlearning.com/

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots

import statsmodels.api as sm
# from statsmodels.gam.api import BSplines, GLMGam

#from sklearn.model_selection import train_test_split
#from sklearn.compose import ColumnTransformer, make_column_selector as selector
#from sklearn.preprocessing import OneHotEncoder, StandardScaler

#from sklearn.linear_model import LinearRegression
#from sklearn.linear_model import RidgeCV
#from sklearn.linear_model import LassoCV
#from sklearn.decomposition import PCA
#from sklearn.cross_decomposition import PLSRegression

# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import KFold, GridSearchCV
# from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import GradientBoostingClassifier

## *Conceptual Exercises*

### (3) Consider the Gini index, classification error, and entropy in a simple classification setting with two classes. Create a single plot that displays each of these quantities as a function of $\hat p_{m1}$. The x-axis should display  $\hat p_{m1}$, ranging from 0 to 1, and the y-axis should display the value of the Gini index, classification error, and entropy.

**Solution**: In a binary classification problem, suppose a given node $ m $ contains observations belonging to two classes, with probabilities:
$$
\hat{p}_{m1} = p, \quad \hat{p}_{m2} = 1 - p.
$$
We can define three impurity measures that quantify how “mixed” the node is:

1. **Classification Error**:
   $$
   E(p) = 1 - \max(p, 1 - p)\quad\text{(8.5) - see pp. 338}
   $$

2. **Gini Index**:
   $$
   G(p) = 2p(1 - p)\quad\text{(8.6) - see pp. 338}
   $$

3. **Entropy (Deviance)**:
   $$
   H(p) = - \big[ p \log_2(p) + (1 - p) \log_2(1 - p) \big]\quad\text{(8.7) - see pp. 339}
   $$

In all indexes it is measured the uncertainty (in bits) in the class distribution. This implies that:  
   - Minimum impurity (0) occurs at $ p = 0 $ or $ p = 1 $.
   - Maximum impurity (1) occurs at $ p = 0.5 $.

These measures behave similarly — all are symmetric and concave, reaching their peak when the two classes are equally probable — but they differ in scale and curvature, which affects tree-splitting decisions.

#### Creating the plot for this problem

In [ ]:
# Range of p values (probability of class 1)
p = np.linspace(0, 1, 500)
eps = 1e-12  # small value to avoid log(0)

# Classification error
classification_error = 1 - np.maximum(p, 1 - p)

# Gini index
gini = 2 * p * (1 - p)

# Entropy (using log base 2)
entropy = - (p * np.log2(p + eps) + (1 - p) * np.log2(1 - p + eps))

# --- Plot ---
plt.figure(figsize=(8, 6))
plt.plot(p, classification_error, label='Classification Error', color='tab:red', lw=2)
plt.plot(p, gini, label='Gini Index', color='tab:blue', lw=2)
plt.plot(p, entropy, label='Entropy', color='tab:green', lw=2)

plt.title('Impurity Measures as a Function of $\hat{p}_{m1}$', fontsize=14)
plt.xlabel(r'$\hat{p}_{m1}$', fontsize=12)
plt.ylabel('Impurity Value', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The figure above shows how **three impurity measures** — *Classification Error*, *Gini Index*, and *Entropy* — vary as a function of the class probability $\hat{p}_{m1}$ in a binary classification setting.

- All three curves are **symmetric around $\hat{p}_{m1} = 0.5$**.
- This symmetry reflects the fact that impurity depends only on *how mixed* the classes are, not on which class is labeled “1” or “2”.
- The impurity is **minimum (0)** when a node is pure, i.e., all samples belong to one class ($\hat{p}_{m1} = 0$ or $1$).
- The impurity is **maximum when $\hat{p}_{m1} = 0.5$**, meaning the node is perfectly mixed.

## *Applied Exercises*- Work in Progress

### (11) This question uses the `Caravan` data set. Data Dictionary at https://liacs.leidenuniv.nl/~puttenpwhvander/library/cc2000/data.html

### (a) Create a training set consisting of the first $1,000$ observations, and a test set consisting of the remaining observations.

In [ ]:
from ISLP import load_data
Caravan = load_data('Caravan')
Caravan.columns

In [ ]:
Caravan.info()
caravan_vars = pd.read_csv("..\\Data\\caravan_vars.csv")

In [ ]:
# --- Split into training and test sets ---
train = Caravan.iloc[:1000, :].copy()
test = Caravan.iloc[1000:, :].copy()

# --- Separate predictors (X) and target (y) ---
X_train = train.drop(columns=["Purchase"])
y_train = train["Purchase"]

X_test = test.drop(columns=["Purchase"])
y_test = test["Purchase"]

# --- Quick validation ---
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Training target distribution:\n", y_train.value_counts(normalize=True))
print("Test target distribution:\n", y_test.value_counts(normalize=True))

### (b) Fit a boosting model to the training set with `Purchase` as the response and the other variables as predictors. Use $1,000$ trees, and a shrinkage value of $0.01$. Which predictors appear to be the most important?

In [ ]:
# Encode Purchase to 0/1
y_train_enc = y_train.astype("category").cat.codes
y_test_enc  = y_test.astype("category").cat.codes

# --- 2) Fit Gradient Boosting Model ---
gb = GradientBoostingClassifier(
    n_estimators=1000,      # number of trees
    learning_rate=0.01,     # shrinkage parameter
    max_depth=3,            # default tree depth
    random_state=42
)
gb.fit(X_train, y_train_enc)

# --- 3) Variable Importance ---
importances = pd.Series(gb.feature_importances_, index=X_train.columns)

# Sort descending
imp_sorted = importances.sort_values(ascending=False).reset_index()
imp_sorted.columns = ["var", "importance"]

# --- 4) Merge with dictionary (left join) ---
imp_merged = pd.merge(imp_sorted, caravan_vars, on="var", how="left")

# --- 5) Display top 15 predictors with description ---
topk = 10
display(imp_merged.head(topk))

# --- 6) Bar plot of top variables ---
top_imp = imp_merged.head(topk).iloc[::-1]  # reverse for barh
plt.figure(figsize=(9, 6))
plt.barh(top_imp["var"], top_imp["importance"], color="tab:blue")
plt.title("Gradient Boosting Feature Importance (Top 15 Predictors)")
plt.xlabel("Normalized Importance")
plt.tight_layout()
plt.show()

### (c) Use the boosting model to predict the response on the test data. Predict that a person will make a purchase if the estimated probability of purchase is greater than $20\%$. Form a confusion matrix. What fraction of the people predicted to make a purchase do in fact make one? How does this compare with the results obtained from applying KNN or logistic regression to this data set?

In [ ]:
# --- 1) Predicted probabilities on test data ---
y_prob = gb.predict_proba(X_test)[:, 1]  # probability of class "Yes"

# --- 2) Classify as 'Yes' if probability > 0.20 ---
y_pred = (y_prob > 0.20).astype(int)

cm_boost = confusion_matrix(y_test_enc, y_pred_boost)
TPb, FPb = cm_boost[1, 1], cm_boost[0, 1]
prec_boost = TPb / (TPb + FPb)
acc_boost = (TPb + cm_boost[0, 0]) / cm_boost.sum()

# ----------------------------------------------
# 3) KNN (k=5)
# ----------------------------------------------
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_std, y_train_enc)

# KNN gives hard predictions; convert to “probabilities” via mean of neighbors
y_prob_knn = knn.predict_proba(X_test_std)[:, 1]
y_pred_knn = (y_prob_knn > 0.20).astype(int)

cm_knn = confusion_matrix(y_test_enc, y_pred_knn)
TPk, FPk = cm_knn[1, 1], cm_knn[0, 1]
prec_knn = TPk / (TPk + FPk)
acc_knn = (TPk + cm_knn[0, 0]) / cm_knn.sum()

# --- 3) Confusion matrix ---
cm = confusion_matrix(y_test_enc, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"]
)
display(cm_df)

# --- 4) Precision: fraction of predicted buyers who actually purchased ---
TP = cm[1, 1]
FP = cm[0, 1]
precision = TP / (TP + FP) if (TP + FP) > 0 else np.nan
print(f"Fraction of predicted purchasers who actually purchased: {precision:.3f}")

# --- 5) show overall accuracy for context ---
accuracy = (TP + cm[0, 0]) / cm.sum()
print(f"Overall accuracy: {accuracy:.3f}")

# ----------------------------------------------
# 4) Display results summary
# ----------------------------------------------
summary = pd.DataFrame({
    "Model": ["Boosting", "KNN (k=5)"],
    "Precision (TP / (TP+FP))": [prec_boost, prec_knn],
    "Accuracy": [acc_boost, acc_knn]
})

display(summary.round(3))


#### Comments

- **Accuracy** is high ($\sim 90\%$) for both models, but this is expected given class imbalance — most customers did not purchase.  
- **Precision** ($\sim 16\%$) is the key metric: both models were able to identify customer's more than **2.5× more likely** to buy compared to the baseline ($\sim 6\%$).
> A precision of $16\%$ may sound low, but in a dataset where only $6\%$ of people buy, it means the model is over **2.5 times better than random** at finding real purchasers a strong result in marketing analytics. The extra $10\%$ comes from *focusing on better prospects* identified by the model — not that the total number of buyers increased, but the *proportion among predicted positives* did. 
- **Boosting** slightly outperforms KNN (0.161 vs. 0.160 precision) and is generally **more stable** for high-dimensional data.  
- **KNN** performs similarly but is more sensitive to scaling and may struggle with so many predictors.  
- Overall, **both models effectively enrich the predicted “Yes” group**, but **Boosting offers better generalization and interpretability** for this task.
